# Predictive Model — Next Day PnL Bucket


This notebook builds a Random Forest classifier to predict whether a trader's next day will be a Big Loss, Small Loss, Small Win, or Big Win.


In [46]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#2e3250',
    'axes.labelcolor'  : '#c9d1d9',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'text.color'       : '#c9d1d9',
    'grid.color'       : '#21262d',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

print("Libraries loaded")

Libraries loaded



## Step 1 — Rebuild the Daily Dataset

We need the same daily aggregated data from the analysis notebook.

In [47]:
# load both raw datasets again
fear_greed = pd.read_csv(r'C:\Users\Hp\Desktop\My Work\primetrade-assignment\data\fear_greed_index.csv')
trader     = pd.read_csv(r'C:\Users\Hp\Desktop\My Work\primetrade-assignment\notebooks\trader_sample.csv')

# convert dates and merge exactly like we did in analysis_charts.ipynb
fear_greed['date'] = pd.to_datetime(fear_greed['date'])
trader['date']     = pd.to_datetime(
    trader['Timestamp IST'], format='%d-%m-%Y %H:%M', dayfirst=True
).dt.normalize()

fg_daily = fear_greed[['date', 'value', 'classification']].rename(columns={
    'value': 'fg_value', 'classification': 'sentiment'
})

merged = trader.merge(fg_daily, on='date', how='inner')
merged['is_win']  = merged['Closed PnL'] > 0
merged['is_long'] = merged['Direction'].isin(['Open Long', 'Close Short', 'Long > Short'])

# aggregate to daily level per trader
daily = (
    merged
    .groupby(['Account', 'date', 'sentiment', 'fg_value'])
    .agg(
        daily_pnl    = ('Closed PnL', 'sum'),
        trades_count = ('Trade ID',   'count'),
        win_trades   = ('is_win',     'sum'),
        avg_size_usd = ('Size USD',   'mean'),
        long_trades  = ('is_long',    'sum'),
    )
    .reset_index()
)

daily['win_rate']   = daily['win_trades']  / daily['trades_count']
daily['long_ratio'] = daily['long_trades'] / daily['trades_count']

print(f"Daily dataset ready: {len(daily):,} rows")

Daily dataset ready: 1,211 rows



## Step 2 — Create the Target Variable (PnL Bucket)

In [48]:
# we convert the continuous daily_pnl into 4 categories
# this turns a regression problem into a classification problem
# which is easier to interpret and explain

daily['pnl_bucket'] = pd.cut(
    daily['daily_pnl'],
    bins   = [-np.inf, -100, 0, 100, np.inf],
    labels = ['Big Loss', 'Small Loss', 'Small Win', 'Big Win']
)

# drop rows where pnl_bucket could not be assigned (should be none but safe to check)
daily = daily.dropna(subset=['pnl_bucket']).copy()



print("PnL bucket distribution:")
print(daily['pnl_bucket'].value_counts().to_string())

PnL bucket distribution:
pnl_bucket
Small Loss    549
Small Win     397
Big Win       228
Big Loss       37


## Step 3 — Prepare Features

In [49]:
# the model needs numbers, but sentiment is text
# LabelEncoder converts text categories to integers
le = LabelEncoder()
daily['sentiment_encoded'] = le.fit_transform(daily['sentiment'])

print("Sentiment encoding mapping:")
for i, label in enumerate(le.classes_):
    print(f"  {i} = {label}")

# features we pass to the model
# fg_value         : the raw fear/greed score (0-100)
# sentiment_encoded: numerical version of sentiment label
# trades_count     : how many trades were made that day
# avg_size_usd     : average position size in dollars
# long_ratio       : how bullish the trader was (0 to 1)
# win_rate         : what fraction of trades were winners

feature_columns = [
    'fg_value',
    'sentiment_encoded',
    'trades_count',
    'avg_size_usd',
    'long_ratio',
    'win_rate',
]

X = daily[feature_columns].fillna(0)
y = daily['pnl_bucket']

print(f"\nFeature matrix shape : {X.shape}")
print(f"Target variable shape : {y.shape}")

Sentiment encoding mapping:
  0 = Extreme Fear
  1 = Extreme Greed
  2 = Fear
  3 = Greed
  4 = Neutral

Feature matrix shape : (1211, 6)
Target variable shape : (1211,)



## Step 4 — Train and Evaluate the Model

In [50]:
# split data into training set (80%) and test set (20%)
# random_state=42 ensures we get the same split every time we run this
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training rows : {len(X_train)}")
print(f"Test rows     : {len(X_test)}")



Training rows : 968
Test rows     : 243


## Step 5 — Model Training

In [51]:
models = {
    "AdaBoostClassifier" : AdaBoostClassifier(),
    "xgboost"            : XGBClassifier(),
    "RandomForestClassifier" : RandomForestClassifier(),
    "LogisticRegression"  : LogisticRegression(),
    "DecisionTreeClassifier" : DecisionTreeClassifier(),
    "SVC" : SVC(),
    "KNN" : KNeighborsClassifier()
}


result = []
for name,model in models.items():


    model.fit(X_train,y_train)


    y_pred = model.predict(X_test)


    accuracy  = accuracy_score(y_test,y_pred)
    matrix= confusion_matrix(y_test,y_pred)
    report = classification_report(y_test,y_pred)

    result.append({
    "Model Name": name,
    'accuracy_score':accuracy,
    })

model_result = pd.DataFrame(result)

print(model_result.sort_values(by="accuracy_score",ascending= False))


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3], got ['Big Loss' 'Big Win' 'Small Loss' 'Small Win']


## Step 6 — Confusion Matrix